In [ ]:
import os
from glob import glob
import geopandas
import pandas
import subprocess
from pathlib import Path
import sys
# !{sys.executable} -m pip install "nismod-snail==0.5.3"

root = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import robyns_libraries.vector_raster_intersections

In [ ]:
# processed_data_path = 'L:\Jamaica\Inputs'
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3/results"
processed_data_path = base_path / "dphil_paper_3/processed_data"
networks_path = base_path / "dphil_common_cross_cutting/common_incoming_data/networks/networks"
damage_curves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/damage_curves"
robyn_libraries_path = base_path / "robyns_libraries"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"

# base_path = 'Z:\\jamaica\\Inputs'
# output_path = 'Z:\\jamaica\\Results'

In [ ]:
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv" 
hazard_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
damage_curves_csv = damage_curves_path / "asset_damage_curve_mapping.csv"
hazard_damage_parameters_csv = damage_curves_path / "hazard_damage_parameters.csv"
damage_results_folder = base_path / "dphil_paper_3/processed_data/direct_damages"


# network_csv = os.path.join(processed_data_path,
#                             "networks",
#                             "network_layers_hazard_intersections_details.csv")
# hazard_csv = os.path.join(processed_data_path,
#                             "coastal_flood_rasters.csv")
# damage_curves_csv = os.path.join(processed_data_path,
#                             "damage_curves",
# #                             "asset_damage_curve_mapping.csv")
# hazard_damage_parameters_csv = os.path.join(processed_data_path,
#                             "damage_curves",
#                             "hazard_damage_parameters.csv")
# damage_results_folder = "direct_damages"

In [ ]:
# Create a path to store intersection outputs
#output_path = os.path.join(output_path,"coastal_flood_intersections")
#if os.path.exists(output_path) == False:
    #os.mkdir(output_path)

In [ ]:
vector_details_csv = data_root /"networks/network_layers.csv"
raster_details_csv = data_root /"networks/hazard_layers.csv"

In [ ]:
# vector_details_csv = os.path.join(base_path,"infrastructure","network_layers.csv")
# raster_details_csv = os.path.join(base_path,"coastal_floods_FN","hazard_layers.csv")

In [ ]:
project_data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
results_directory = Path(output_path)
results_directory.mkdir(parents=True, exist_ok=True)

network_layers_input_file = project_data_root / "networks/network_layers.csv"
network_layers_table = pandas.read_csv(network_layers_input_file)
network_layers_table["path"] = network_layers_table["path"].str.replace(
    r"^networks/", "networks/networks/", regex=True
)
network_layers_output_file = results_directory / "network_layers_fixed_for_intersections.csv"
network_layers_table.to_csv(network_layers_output_file, index=False)

hazard_layers_input_file = project_data_root / "networks/hazard_layers.csv"
hazard_layers_table = pandas.read_csv(hazard_layers_input_file)
hazard_layers_table["fname"] = hazard_layers_table["path"]  # required by vector_raster_intersections.py
hazard_layers_output_file = results_directory / "hazard_layers_fixed_for_intersections.csv"
hazard_layers_table.to_csv(hazard_layers_output_file, index=False)

vector_details_csv = network_layers_output_file
raster_details_csv = hazard_layers_output_file

print("Network layers file:", vector_details_csv)
print("Hazard layers file:", raster_details_csv)


In [ ]:
run_intersections = True  # Set to True is you want to run this process
if run_intersections is True:
    args = [
            "python",
            str(base_path / "robyns_libraries/vector_raster_intersections.py"),
            f"{vector_details_csv}",
            f"{raster_details_csv}",
            f"{output_path}"
            ]
    print ("* Start the processing of vector-raster intersections")
    print (args)
    subprocess.run(args)
print ("* Done with the processing of vector-raster intersections")

In [ ]:
# run_intersections = True  # Set to True is you want to run this process
# if run_intersections is True:
#     args = [
#             "python",
#             vector_raster_intersections.py",
#             f"{vector_details_csv}",
#             f"{raster_details_csv}",
#             f"{output_path}"
#             ]
#     print ("* Start the processing of vector-raster intersections")
#     print (args)
#     subprocess.run(args)
# print ("* Done with the processing of vector-raster intersections")

In [ ]:
# file_name = os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.geoparquet")
# df = geopandas.read_parquet(file_name)
# df



hazard_layers_name = Path(raster_details_csv).stem
file_name = Path(output_path) / f"airport_polygon_splits__{hazard_layers_name}__areas.geoparquet"
df = geopandas.read_parquet(file_name)
df

In [ ]:
# df.to_file(os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.gpkg"), layer="area", driver="GPKG")

gpkg_name = Path(output_path) / f"airport_polygon_splits__{hazard_layers_name}__areas.gpkg"
df.to_file(gpkg_name, layer="area", driver="GPKG")


In [ ]:
import subprocess

script_path = base_path / "scripts/analysis/damage_calculations.py"
hazard_layers_name = Path(raster_details_csv).stem
damage_results_folder = Path(output_path) / "direct_damages"
damage_results_folder.mkdir(parents=True, exist_ok=True)

sensitivity_csv = Path(output_path) / "sensitivity_parameters.csv"
pandas.DataFrame(
    [{"cost_uncertainty_parameter": 0.0, "damage_uncertainty_parameter": 0.0}]
).to_csv(sensitivity_csv, index=False)

asset_data_details = pandas.read_csv(network_csv)

for asset_info in asset_data_details.itertuples():
    asset_file_from_data_root = data_root / asset_info.path
    asset_file_with_networks_prefix = data_root / "networks" / asset_info.path

    if asset_file_from_data_root.exists():
        asset_gpkg_file = asset_file_from_data_root
    elif asset_file_with_networks_prefix.exists():
        asset_gpkg_file = asset_file_with_networks_prefix
    else:
        raise FileNotFoundError(
            "Asset file not found at either expected location:\n"
            f"{asset_file_from_data_root}\n"
            f"{asset_file_with_networks_prefix}"
        )
    intersection_file = Path(output_path) / f"{asset_info.asset_gpkg}_splits__{hazard_layers_name}__{asset_info.asset_layer}.geoparquet"
    output_file = damage_results_folder / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}" / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    args = [
        "python", str(script_path),
        "--network-csv", str(network_csv),
        "--hazard-csv", str(hazard_csv),
        "--sensitivity-csv", str(sensitivity_csv),
        "--sensitivity-id", "0",
        "--asset-gpkg-file", str(asset_gpkg_file),
        "--asset-gpkg-label", str(asset_info.asset_gpkg),
        "--asset-layer", str(asset_info.asset_layer),
        "--damage-curve-mapping-csv", str(damage_curves_csv),
        "--damage-threshold-uplift-csv", str(hazard_damage_parameters_csv),
        "--damage-curves-dir", str(damage_curves_path),
        "--intersection", str(intersection_file),
        "--output-path", str(output_file),
    ]
    print(args)
    run_result = subprocess.run(args, capture_output=True, text=True)
    if run_result.returncode != 0:
        if "No objects to concatenate" in (run_result.stderr or ""):
            print(f"Skipping {asset_info.asset_gpkg} {asset_info.asset_layer}: no intersecting damages")
            continue
        print(run_result.stdout)
        print(run_result.stderr)
        run_result.check_returncode()

print("Finished direct damage calculations")


In [ ]:
# """Next we call the summary scripts
# """
# args = [
#         "python",
#         "damage_calculations.py",
#         f"{damage_results_folder}",
#         f"{network_csv}",
#         f"{hazard_csv}",
#         f"{damage_curves_csv}",
#         f"{hazard_damage_parameters_csv}",
#         "0","0","0"
#         ]
# print ("* Start the processing of summarising damage results")
# print (args)
# subprocess.check_output(args)

In [ ]:
damage_results_folder = os.path.join(output_path, "direct_damages")
mangrove_flood_damage_columns = ["coastal_flood_fn_mg_rp_25",
                       "coastal_flood_fn_mg_rp_100",
                       "coastal_flood_fn_mg_rp_500"]
nomangrove_flood_damage_columns = ["coastal_flood_fn_nomg_rp_25",
                       "coastal_flood_fn_nomg_rp_100",
                       "coastal_flood_fn_nomg_rp_500"]
difference_columns = ["coastal_flood_diff_rp_25",
                       "coastal_flood_diff_rp_100",
                       "coastal_flood_diff_rp_500"]
flood_damage_columns = mangrove_flood_damage_columns + nomangrove_flood_damage_columns + difference_columns 

In [ ]:
jamaica_crs = 3448

asset_data_details = pandas.read_csv(network_csv)
damage_totals = [] # List object to assemble many dataframes 
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path) #read in the files using the .parquet from Raghav's code
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index() # calculate asset level damages = .groupby(node_id).sum()
            df_path = os.path.join(processed_data_path,
                                   f"{asset_path}")
            df_geom = geopandas.read_file(df_path, layer = asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            print(df)
            df.to_file(os.path.join(output_path,
                                    'damage_estimates', 
                                    f'{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg'),
                                    driver='GPKG')
            df['sector'] = asset_gpkg
            df['layer'] = asset_layer
            df = df.groupby(['sector','layer']).sum(flood_damage_columns).reset_index()
            damage_totals.append(df) # Add things to list

# Convert list of dataframes to 1 dataframe by concatenation
damage_totals = pandas.concat(damage_totals,axis=0,ignore_index=True)
damage_totals.to_csv(os.path.join(output_path,
                                    'damage_estimates', 
                                    'asset_damages_groupedby.csv'))

In [ ]:
print(asset_gpkg)

In [ ]:


            # Calculate total sector damages with and without mangroves for each return period

# total_sector_damages_mg_rp_25 = sum(coastal_flood_fn_mg_rp_25)

In [ ]:
# buffer the mangroves by 1km; intersect with all the assets and output a list of mangrove ID and asset ID OR do nearest neighbour

jamaica_crs = 3448

FN_Mangroves = geopandas.read_file(os.path.join(processed_data_path,"Forces of nature mangroves", 'mangroves.shp'))[["ID","geometry"]]
FN_Mangroves = FN_Mangroves.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system

#FN_Mangroves["geometry"] = FN_Mangroves.buffer(distance=1000)

asset_data_details = pandas.read_csv(network_csv)
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path)
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index()
            df_path = os.path.join(processed_data_path,
                                   f"{asset_path}")
            df_geom = geopandas.read_file(df_path, layer = asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            #mangroves_asset_intersection = FN_Mangroves[["ID","geometry"]] \
            #.overlay(df,how="intersection",keep_geom_type=False)
            #print(mangroves_asset_intersection[["ID",asset_id]])
        
        #for loop to calculate all the mangrove ID distances from an asset ID
        
        for asset_info in asset_data_details.itertuples():
            asset_path = asset_info.path
            asset_gpkg = asset_info.asset_gpkg
            asset_layer = asset_info.asset_layer
            asset_id = asset_info.asset_id_column
            df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        

In [ ]:
asset_data_details = pandas.read_csv(network_csv)
mapping_result = []
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(processed_data_path,
                               f"{asset_path}")
        df = geopandas.read_file(df_path, layer = asset_layer)
        df = df.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
        #mangroves_asset_intersection = FN_Mangroves[["ID","geometry"]] \
        #.overlay(df,how="intersection",keep_geom_type=False)
        #print(mangroves_asset_intersection[["ID",asset_id]])
        #intersect buffered mangrove with the output of the above loop - with the asset id 
        


In [ ]:
# direct_damages_file = os.path.join(hazard_asset_intersection_path,
  #                              f"{asset_info.asset_gpkg}_with_coastal_{asset_info.asset_layer}.parquet")
   #     if os.path.isfile(hazard_intersection_file) is True:
    #        hazard_df = geopandas.read_parquet(hazard_intersection_file)

In [ ]:
#run_intersections = True  # Set to True is you want to run this process
#if run_intersections is True:
    #args = [
#             "python",
#             "vector_raster_intersections.py",
#             f"{vector_details_csv}",
#             f"{raster_details_csv}",
#             f"{output_path}"
#             ]
#     print ("* Start the processing of vector-raster intersections")
#     print (args)
#     subprocess.run(args)
# print ("* Done with the processing of vector-raster intersections")

In [ ]:
# df.to_file(os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.gpkg"), layer="area", driver="GPKG")